# CZSO IKZ - manifest + excluded export

Tento notebook vytvoří jeden nový XLSX soubor se dvěma listy:

- `manifest` - obsah z `ikz_a_to_z_manifest.xlsx`
- `excluded` - všechny datasety, které jsou v `selected_relevant_catalogue`, ale nejsou ve finálním manifestu

Notebook je určený k vložení do posledního Python bundle.


In [1]:
from pathlib import Path
import re
import pandas as pd

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / 'czso_ikz_a_to_z_output'
MANIFEST_XLSX = OUTPUT_DIR / 'ikz_a_to_z_manifest.xlsx'
MANIFEST_CSV = OUTPUT_DIR / 'ikz_a_to_z_manifest.csv'
SELECTED_CATALOGUE_CSV = OUTPUT_DIR / 'selected_relevant_catalogue.csv'
SELECTED_CATALOGUE_XLSX = OUTPUT_DIR / 'selected_relevant_catalogue.xlsx'

# fallback, pokud selected_relevant_catalogue leží vedle notebooku
ALT_SELECTED_CATALOGUE_CSV = BASE_DIR / 'selected_relevant_catalogue.csv'
ALT_SELECTED_CATALOGUE_XLSX = BASE_DIR / 'selected_relevant_catalogue.xlsx'

OUTPUT_XLSX = OUTPUT_DIR / 'ikz_a_to_z_manifest_with_excluded.xlsx'

print('BASE_DIR   :', BASE_DIR)
print('OUTPUT_DIR :', OUTPUT_DIR)
print('MANIFEST   :', MANIFEST_XLSX if MANIFEST_XLSX.exists() else MANIFEST_CSV)
print('SELECTED   :', SELECTED_CATALOGUE_CSV if SELECTED_CATALOGUE_CSV.exists() else (SELECTED_CATALOGUE_XLSX if SELECTED_CATALOGUE_XLSX.exists() else (ALT_SELECTED_CATALOGUE_CSV if ALT_SELECTED_CATALOGUE_CSV.exists() else ALT_SELECTED_CATALOGUE_XLSX)))
print('OUTPUT_XLSX:', OUTPUT_XLSX)


BASE_DIR   : C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle
OUTPUT_DIR : C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle\czso_ikz_a_to_z_output
MANIFEST   : C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle\czso_ikz_a_to_z_output\ikz_a_to_z_manifest.xlsx
SELECTED   : C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle\czso_ikz_a_to_z_output\selected_relevant_catalogue.csv
OUTPUT_XLSX: C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle\czso_ikz_a_to_z_output\ikz_a_to_z_manifest_with_excluded.xlsx


In [2]:
def _read_table(preferred_xlsx: Path, preferred_csv: Path) -> pd.DataFrame:
    if preferred_xlsx.exists():
        try:
            xls = pd.ExcelFile(preferred_xlsx)
            sheet = xls.sheet_names[0]
            return pd.read_excel(preferred_xlsx, sheet_name=sheet)
        except Exception as e:
            print(f'Varování: nepodařilo se načíst {preferred_xlsx}: {e}')
    if preferred_csv.exists():
        return pd.read_csv(preferred_csv)
    raise FileNotFoundError(f'Nenalezen soubor: {preferred_xlsx} ani {preferred_csv}')


def _read_selected_catalogue() -> pd.DataFrame:
    candidates = [
        (SELECTED_CATALOGUE_XLSX, SELECTED_CATALOGUE_CSV),
        (ALT_SELECTED_CATALOGUE_XLSX, ALT_SELECTED_CATALOGUE_CSV),
    ]
    errors = []
    for xlsx_path, csv_path in candidates:
        try:
            return _read_table(xlsx_path, csv_path)
        except FileNotFoundError as e:
            errors.append(str(e))
    raise FileNotFoundError(' | '.join(errors))


_SPACE_RE = re.compile(r'\s+')


def _norm_text(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value).replace('﻿', '').replace('"', '').strip().lower()
    return _SPACE_RE.sub(' ', s)


def _title_series(df: pd.DataFrame) -> pd.Series:
    if 'dataset_title' in df.columns:
        return df['dataset_title']
    if 'title' in df.columns:
        return df['title']
    return pd.Series([''] * len(df), index=df.index)


def _build_row_key(df: pd.DataFrame) -> pd.Series:
    dataset_id = df['dataset_id'].astype(str).map(_norm_text) if 'dataset_id' in df.columns else pd.Series([''] * len(df), index=df.index)
    title = _title_series(df).map(_norm_text)
    page = df['page'].map(_norm_text) if 'page' in df.columns else pd.Series([''] * len(df), index=df.index)
    return dataset_id + '||' + title + '||' + page


def _guess_exclusion_reason(row: pd.Series) -> str:
    if str(row.get('excluded_historical', '')).lower() == 'true':
        return 'historické / vyřazené jako staré'
    latest_year = row.get('latest_year_hint')
    try:
        if pd.notna(latest_year) and float(latest_year) < 2020:
            return 'pravděpodobně historický dataset (<2020)'
    except Exception:
        pass
    return 'není ve finálním manifestu'


def _autosize_excel(writer, sheet_name: str, df: pd.DataFrame) -> None:
    ws = writer.sheets[sheet_name]
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions
    for idx, col in enumerate(df.columns, start=1):
        values = [str(col)] + [str(v) for v in df[col].head(200).tolist()]
        width = min(max(len(v) for v in values) + 2, 60)
        ws.column_dimensions[ws.cell(row=1, column=idx).column_letter].width = width


In [3]:
manifest_df = _read_table(MANIFEST_XLSX, MANIFEST_CSV).copy()
selected_df = _read_selected_catalogue().copy()

manifest_df['__row_key__'] = _build_row_key(manifest_df)
selected_df['__row_key__'] = _build_row_key(selected_df)

manifest_keys = set(manifest_df['__row_key__'].dropna())
excluded_df = selected_df.loc[~selected_df['__row_key__'].isin(manifest_keys)].copy()

if 'exclusion_reason' not in excluded_df.columns:
    excluded_df['exclusion_reason'] = excluded_df.apply(_guess_exclusion_reason, axis=1)

preferred_cols = [
    'dataset_id', 'dataset_title', 'title', 'rodina_datasetu', 'relevance_pro_IKZ', 'relevance_rank',
    'hlavni_skupina', 'detailni_skupina', 'uzemni_uroven', 'zdroj_typ', 'periodicita_text',
    'latest_year_hint', 'excluded_historical', 'category', 'order', 'page', 'exclusion_reason'
]
excluded_cols = [c for c in preferred_cols if c in excluded_df.columns] + [c for c in excluded_df.columns if c not in preferred_cols + ['__row_key__']]
excluded_df = excluded_df[excluded_cols]

manifest_export_df = manifest_df.drop(columns=['__row_key__'], errors='ignore')

print('Manifest rows :', len(manifest_export_df))
print('Selected rows :', len(selected_df))
print('Excluded rows :', len(excluded_df))

display(manifest_export_df.head(3))
display(excluded_df.head(10))


Manifest rows : 273
Selected rows : 732
Excluded rows : 383


,order,category,category_reason,relevance_pro_ikz,relevance_rank,dataset_id,dataset_title,rodina_datasetu,hlavni_skupina,detailni_skupina,...,tail_csv,metadata_json,page,error,territorial_scope,territorial_scope_label,territorial_rank,territorial_evidence,family_sort_key,variant_year
0,1,A,obsahuje cílové roky (data) a má primary key,jádrová pro IKŽ,0,200068,Dokončené byty v obcích,Dokončené byty v obcích,Bydlení a technická infrastruktura,bytová výstavba a stavebnictví,...,previews\A_200068_Dokoncene_byty_v_obcich_tail...,previews\A_200068_Dokoncene_byty_v_obcich_meta...,https://csu.gov.cz/docs/107508/b8162c4b-9666-e...,Archiv obsahuje 1 čitelných souborů; profilová...,obec,obec,0,uzemi_cis/vuzemi_cis=43,dokoncenebytyvobcich,2024
1,2,C,"má primary key, ale neobsahuje cílové roky (data)",jádrová pro IKŽ,0,190037,Zařízení sociálních služeb podle obcí,Zařízení sociálních služeb podle obcí,Zdraví a sociální služby,kapacity sociálních služeb,...,previews\C_190037_Zarizeni_socialnich_sluzeb_p...,previews\C_190037_Zarizeni_socialnich_sluzeb_p...,https://csu.gov.cz/docs/107508/4b842027-3f33-1...,encoding=utf-8-sig | časové sloupce detekovány...,obec,obec,0,primary_key_column=obec_kod,zarizenisocialnichsluzebpodleobci,2024
2,3,C,"má primary key, ale neobsahuje cílové roky (data)",jádrová pro IKŽ,0,130141r23,Pohyb obyvatel - rok 2022,Pohyb obyvatel,Demografie a populace,přirozená měna a migrace,...,previews\C_130141r23_Pohyb_obyvatel_-_rok_2022...,previews\C_130141r23_Pohyb_obyvatel_-_rok_2022...,https://csu.gov.cz/docs/107508/e40b9b8d-ff26-8...,encoding=utf-8-sig | časové sloupce detekovány...,obec,obec,0,"uzemi_cis/vuzemi_cis=43,65",pohybobyvatel,2022


,dataset_id,dataset_title,title,rodina_datasetu,relevance_pro_IKZ,relevance_rank,hlavni_skupina,detailni_skupina,uzemni_uroven,zdroj_typ,...,provider,modified,start,end,description,keywords_all,dataset_iri,spatial,periodicity,_input_order
18,130140or,Úmrtnostní tabulky pro správní obvody obcí s r...,Úmrtnostní tabulky pro správní obvody obcí s r...,Úmrtnostní tabulky pro správní obvody obcí s r...,podpůrná / kontextová,1,Zdraví a sociální služby,úmrtnost a délka života,nespecifikováno / mix,běžná statistika ČSÚ,...,Český statistický úřad,2025-05-30,2001-01-01,2024-12-31,Podrobné úmrtnostní tabulky podle pohlaví pro ...,"úmrtnostní tabulky, naděje dožití, střední dél...",https://vdb.czso.cz/pll/eweb/lkod_ld.datova_sa...,https://linked.cuzk.cz/resource/ruian/ST/1,R/P1Y,252
27,130140ok,Úmrtnostní tabulky pro okresy,Úmrtnostní tabulky pro okresy,Úmrtnostní tabulky pro okresy,podpůrná / kontextová,1,Zdraví a sociální služby,úmrtnost a délka života,okres,běžná statistika ČSÚ,...,Český statistický úřad,2025-05-30,2001-01-01,2024-12-31,Podrobné úmrtnostní tabulky podle pohlaví pro ...,"úmrtnostní tabulky, naděje dožití, střední dél...",https://vdb.czso.cz/pll/eweb/lkod_ld.datova_sa...,https://linked.cuzk.cz/resource/ruian/ST/1,R/P1Y,254
28,res_pf_nace,Registr ekonomických subjektů - seznam právníc...,Registr ekonomických subjektů - seznam právníc...,Registr ekonomických subjektů - seznam právníc...,podpůrná / kontextová,1,"Práce, příjmy a ekonomika",ekonomické subjekty a podnikatelská struktura,nespecifikováno / mix,běžná statistika ČSÚ,...,Český statistický úřad,2021-09-16,NaN,NaN,Seznam právních forem a činností ekonomických ...,"RES, organizační statistika, Registr ekonomick...",https://vdb.czso.cz/pll/eweb/lkod_ld.datova_sa...,https://linked.cuzk.cz/resource/ruian/ST/1,R/P0.5M,235
29,res_data,Registr ekonomických subjektů - seznam subjektů,Registr ekonomických subjektů - seznam subjektů,Registr ekonomických subjektů - seznam subjektů,podpůrná / kontextová,1,"Práce, příjmy a ekonomika",ekonomické subjekty a podnikatelská struktura,nespecifikováno / mix,běžná statistika ČSÚ,...,Český statistický úřad,2021-09-16,NaN,NaN,Seznam ekonomických subjektů,"RES, organizační statistika, Registr ekonomick...",https://vdb.czso.cz/pll/eweb/lkod_ld.datova_sa...,https://linked.cuzk.cz/resource/ruian/ST/1,R/P0.5M,236
48,170241-17,Obyvatelstvo v základních sídelních jednotkách...,Obyvatelstvo v základních sídelních jednotkách...,Obyvatelstvo v základních sídelních jednotkách...,jádrová pro IKŽ,0,Demografie a populace,základní populační stav,podobecní úroveň (část obce/ZSJ),SLDB 2011 / historické sčítání,...,Český statistický úřad,2024-06-28,2011-03-26,2011-03-26,Datová sada obsahuje statistické údaje o počtu...,"Počet obyvatel, obyvatelstvo, obvyklý pobyt, z...",https://vdb.czso.cz/pll/eweb/lkod_ld.datova_sa...,https://linked.cuzk.cz/resource/ruian/ST/1,NEVER,37
50,140133q25,Počty ekonomických subjektů podle odvětví přev...,Počty ekonomických subjektů podle odvětví přev...,Počty ekonomických subjektů podle odvětví přev...,jádrová pro IKŽ,0,"Práce, příjmy a ekonomika",ostatní,obec,běžná statistika ČSÚ,...,Český statistický úřad,2025-04-15,2025-01-01,2025-03-31,Datová sada obsahuje počty ekonomických subjek...,"ekonomický subjekt, zjištěná aktivita, registr...",https://vdb.czso.cz/pll/eweb/lkod_ld.datova_sa...,https://linked.cuzk.cz/resource/ruian/ST/1,R/P3M,101
51,140131q25,Počty ekonomických subjektů podle vybraných pr...,Počty ekonomických subjektů podle vybraných pr...,Počty ekonomických subjektů podle vybraných pr...,jádrová pro IKŽ,0,"Práce, příjmy a ekonomika",ostatní,obec,běžná statistika ČSÚ,...,Český statistický úřad,2025-04-15,2025-01-01,2025-03-31,Datová sada obsahuje počty registrovaných ekon...,"ekonomický subjekt, zjištěná aktivita, registr...",https://vdb.czso.cz/pll/eweb/lkod_ld.datova_sa...,https://linked.cuzk.cz/resource/ruian/ST/1,R/P3M,105
167,130141r20,"Pohyb obyvatel za ČR, kraje, okresy, SO ORP a ...","Pohyb obyvatel za ČR, kra

In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    manifest_export_df.to_excel(writer, sheet_name='manifest', index=False)
    excluded_df.to_excel(writer, sheet_name='excluded', index=False)
    _autosize_excel(writer, 'manifest', manifest_export_df)
    _autosize_excel(writer, 'excluded', excluded_df)

print('Hotovo.')
print('Vytvořen soubor:', OUTPUT_XLSX)


Hotovo.
Vytvořen soubor: C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle\czso_ikz_a_to_z_output\ikz_a_to_z_manifest_with_excluded.xlsx
